# Task 3 — Reranking for Beyond-Accuracy

## Task 3.1 — Rerankers

Task 2 evaluates the lists the models already return. A reranker starts from those scores and builds a new top-10, with an explicit weight on a beyond-accuracy property the original ranking does not optimise. Four rerankers are implemented here.

* **Diversification** — maximal marginal relevance, so the genres inside one list repeat less.
* **Calibration** — Steck's greedy calibration, so the genre mix of the list stays close to the user's own training history.
* **Item-side fairness** — the same greedy form, with a target that spreads slots across the head, the mid catalogue and the tail.
* **User-side fairness** — the gender gap in NDCG. The group that is ahead on the validation split gives up some of that lead. The other group keeps the model's own list.

The candidate pool is the model's top 100. The weight λ is chosen on the validation split and then frozen for the test lists.

### How to run

Run top to bottom in the `recsys` kernel. The notebook reads the score matrices and the splits written by `Task1.ipynb`, and it checks the base lists against `runs/tables/table_2_2_selected.csv`. Nothing is trained. Tables and the figure go to `runs/tables/`. The reranked test lists go to `runs/rerank/`.

In [ ]:
# =============================================================================
# 3.1.0  Paths, side information, score matrices
# =============================================================================
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "RecBole_DSAIT4335").is_dir())
RUNS = PROJECT / "runs"
DATA = PROJECT / "RecBole_DSAIT4335" / "dataset" / "ml-100k"
K = 10
POOL = 100
BUDGET = 0.02
GRID = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9]
ALPHA_GENRE = 0.01
ALPHA_ITEM = 0.1
TARGET_ITEM = np.full(3, 1.0 / 3.0)

MODELS = [
    ("tuned-EASE", "EASE"),
    ("tuned-SLIMElastic", "SLIMElastic"),
    ("tuned-NGCF", "NGCF"),
    ("tuned-NeuMF", "NeuMF"),
    ("tuned-BPR", "BPR"),
]


def ground_truth(split):
    frame = pd.read_csv(RUNS / "splits" / f"ml-100k.{split}.tsv", sep="\t", dtype=str)
    truth = {}
    for user, item in zip(frame.iloc[:, 0], frame.iloc[:, 1]):
        truth.setdefault(user, set()).add(item)
    return truth


GT_TEST = ground_truth("test")
GT_VALID = ground_truth("valid")

_train = pd.read_csv(RUNS / "splits" / "ml-100k.train.tsv", sep="\t", dtype=str)
_counts = _train.iloc[:, 1].value_counts()
POP_PROB = (_counts / int(_counts.sum())).to_dict()
_order = sorted(POP_PROB, key=POP_PROB.get, reverse=True)
_head_n = max(1, int(round(0.1 * len(POP_PROB))))
_mid_n = int(round(0.30 * len(POP_PROB)))
HEAD = set(_order[:_head_n])
MID = set(_order[_head_n:_mid_n])
TAIL = set(_order[_mid_n:])

_items = pd.read_csv(DATA / "ml-100k.item", sep="\t", dtype=str)
_idc = [c for c in _items.columns if c.startswith("item_id")][0]
_clc = [c for c in _items.columns if c.startswith("class")][0]
GENRES = {
    str(i): frozenset(str(c).split()) if isinstance(c, str) and c else frozenset()
    for i, c in zip(_items[_idc], _items[_clc])
}
CATALOG = set(GENRES)
GENRE_NAMES = sorted({g for gs in GENRES.values() for g in gs})
GENRE_INDEX = {g: i for i, g in enumerate(GENRE_NAMES)}

_users = pd.read_csv(DATA / "ml-100k.user", sep="\t", dtype=str)
_ucol = [c for c in _users.columns if c.startswith("user_id")][0]
_gcol = [c for c in _users.columns if c.startswith("gender")][0]
GENDER = dict(zip(_users[_ucol].astype(str), _users[_gcol].astype(str)))


def load_scores(run_id):
    with np.load(RUNS / "scores" / f"{run_id}.npz") as npz:
        return {
            "test": npz["scores_test"].astype(np.float64),
            "valid": npz["scores_valid"].astype(np.float64),
            "users_test": np.asarray(npz["users_test"], dtype=str),
            "users_valid": np.asarray(npz["users_valid"], dtype=str),
            "items": np.asarray(npz["items"], dtype=str),
        }


SCORES = {run_id: load_scores(run_id) for run_id, _ in MODELS}
ITEMS = SCORES["tuned-EASE"]["items"]
USERS_TEST = SCORES["tuned-EASE"]["users_test"]
USERS_VALID = SCORES["tuned-EASE"]["users_valid"]
for _run, _pack in SCORES.items():
    assert np.array_equal(_pack["items"], ITEMS), _run
    assert np.array_equal(_pack["users_test"], USERS_TEST), _run
    assert np.array_equal(_pack["users_valid"], USERS_VALID), _run

ITEM_COL = {tok: j for j, tok in enumerate(ITEMS)}
GENRE_BIN = np.zeros((len(ITEMS), len(GENRE_NAMES)), dtype=np.float64)
for _j, _tok in enumerate(ITEMS):
    for _g in GENRES.get(_tok, ()):
        GENRE_BIN[_j, GENRE_INDEX[_g]] = 1.0
_size = GENRE_BIN.sum(1, keepdims=True)
_size[_size == 0] = 1.0
GENRE_DIST = GENRE_BIN / _size

_inter = GENRE_BIN @ GENRE_BIN.T
_card = GENRE_BIN.sum(1)
_union = _card[:, None] + _card[None, :] - _inter
SIM = np.divide(_inter, _union, out=np.zeros_like(_inter), where=_union > 0)
SIM[(_card[:, None] == 0) & (_card[None, :] == 0)] = 1.0

GROUP_INDEX = np.full(len(ITEMS), 2, dtype=int)
for _j, _tok in enumerate(ITEMS):
    if _tok in HEAD:
        GROUP_INDEX[_j] = 0
    elif _tok in MID:
        GROUP_INDEX[_j] = 1
GROUP_DIST = np.eye(3, dtype=np.float64)[GROUP_INDEX]

_train_items = {}
for _u, _i in zip(_train.iloc[:, 0], _train.iloc[:, 1]):
    _train_items.setdefault(_u, []).append(_i)


def _smoothed(counts, alpha):
    z = counts + alpha
    return z / z.sum()


PROFILE = {}
for _u, _its in _train_items.items():
    cols = [ITEM_COL[i] for i in _its if i in ITEM_COL]
    counts = GENRE_DIST[cols].sum(0) if cols else np.zeros(len(GENRE_NAMES))
    PROFILE[_u] = _smoothed(counts, ALPHA_GENRE)
PROFILE_TEST = np.stack([PROFILE[u] for u in USERS_TEST])
PROFILE_VALID = np.stack([PROFILE[u] for u in USERS_VALID])

print(
    f"catalog {len(CATALOG)} items | genres {len(GENRE_NAMES)} | "
    f"head {_head_n} / mid {len(MID)} / tail {len(TAIL)} | "
    f"{len(USERS_TEST)} users | pool {POOL}"
)

### The four rerankers

Every method sees the same candidates: the model's top 100, scored by that model's relevance after a per-user min–max over the legal candidates. That is the scaling used for the weighted hybrid in Task 1.3. Items the user has already seen are masked in the exported matrices. The list is grown greedily to length 10. λ is the weight on the beyond-accuracy term. At λ = 0 every method returns the original top-10, because the penalty term is then ignored and the pool is already in score order.

**Diversification (MMR).** Maximal marginal relevance (Carbonell and Goldstein, 1998). The next item maximises

\[
(1-\lambda)\, r(u,i) - \lambda \max_{j \in L} \mathrm{sim}(i,j)
\]

where \(\mathrm{sim}\) is the Jaccard similarity of the two genre sets, the complement of the distance behind ILD in Task 2.1. The first pick is still the model's top item: on an empty list the similarity penalty is zero. Later picks pay for sharing genres with what is already there.

**Calibration (Steck, 2018).** The target \(p_u\) is the genre distribution of the user's training history. Each training movie spreads its mass uniformly over the genres it carries. \(q\) is the same average over the movies already chosen. The next item maximises

\[
(1-\lambda)\, r(u,i) - \lambda\, \mathrm{KL}\big(p_u \,\|\, q(L \cup \{i\})\big)
\]

with the divergence in bits. Both distributions are additively smoothed with \(\alpha = 0.01\), so a genre that has not appeared yet does not make the KL infinite. Validation and test interactions are not part of \(p_u\).

**Item-side fairness.** The same update, over popularity instead of genre, and with one target for every user. A movie is head, mid or tail under the cuts from Task 2.5: the most popular 10% of training items, the next 20%, and the rest. A movie with no training interactions is counted as tail. The target is uniform, \((1/3, 1/3, 1/3)\), so the three bands get the same number of slots. The base lists do not look like that: EASE puts about 88% of its top-10 on the head. Smoothing is \(\alpha = 0.1\). The pool still binds the method. A tail movie can be promoted only when the model already placed it inside the top 100.

**User-side fairness.** The groups are the genders in `ml-100k.user`. On the validation split one gender has the higher mean NDCG under the base list. A reranker that sees scores, and not the held-out labels, has no way to raise the lower of those two means: the base top-10 is already the list that maximises the model's own scores. The gap is closed from the side that is ahead. Users in the lower group keep the base list. Users in the higher group receive an MMR list, and λ is the grid point that makes \(|\mathrm{NDCG}_F - \mathrm{NDCG}_M|\) smallest. Which group is higher is read from that model's base validation NDCG and then kept for the test lists.

### Choosing λ

The grid is \(0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.70, 0.90\). For each model and each method the value is chosen on validation, inside a fixed accuracy budget: validation NDCG may fall by at most 0.02 from that model's own base list. Inside the budget, diversification takes the highest ILD, calibration the lowest genre KL, item-side fairness the lowest exposure KL, and user-side fairness the smallest gender gap. A tie goes to the smaller λ. If nothing on the grid improves the target inside the budget, the base list is kept.

Relevance and Jaccard similarity both lie in \([0, 1]\). The two KL terms are in bits, so the same numeric λ is a different trade-off for the calibration and item-fairness rerankers than it is for MMR. The grid is searched separately for each method, which is what makes the selected points comparable.

In [ ]:
# =============================================================================
# 3.1.1  Rerankers and the metrics they are judged on
# =============================================================================
def candidate_pool(scores):
    """Per-user min-max relevance, and the top-POOL item columns in score order."""
    finite = np.isfinite(scores)
    masked = np.where(finite, scores, np.nan)
    low = np.nanmin(masked, axis=1, keepdims=True)
    high = np.nanmax(masked, axis=1, keepdims=True)
    span = np.where(high > low, high - low, 1.0)
    relevance = (masked - low) / span
    filled = np.where(finite, relevance, -np.inf)
    pool = np.argsort(-filled, axis=1, kind="stable")[:, :POOL]
    return relevance, pool


def mmr_lists(pool, relevance, lam):
    """Greedy MMR. lam = 0 returns the pool order, which is the base top-k."""
    if lam == 0:
        return pool[:, :K].copy()
    out = np.empty((pool.shape[0], K), dtype=int)
    for u in range(pool.shape[0]):
        cands = pool[u]
        rel = relevance[u, cands]
        local = SIM[np.ix_(cands, cands)]
        max_sim = np.zeros(POOL)
        alive = np.ones(POOL, dtype=bool)
        for t in range(K):
            score = (1.0 - lam) * rel - lam * max_sim
            score[~alive] = -np.inf
            j = int(np.argmax(score))
            out[u, t] = cands[j]
            alive[j] = False
            max_sim = np.maximum(max_sim, local[j])
    return out


def kl_lists(pool, relevance, features, target, lam, alpha):
    """Greedy relevance-vs-KL. `target` is (D,) or one row per user."""
    if lam == 0:
        return pool[:, :K].copy()
    n_users, dim = pool.shape[0], features.shape[1]
    shared = target.ndim == 1
    out = np.empty((n_users, K), dtype=int)
    for u in range(n_users):
        cands = pool[u]
        rel = relevance[u, cands]
        feats = features[cands]
        desired = target if shared else target[u]
        counts = np.zeros(dim)
        alive = np.ones(POOL, dtype=bool)
        for t in range(K):
            proposed = counts + feats
            q = (proposed + alpha) / (proposed.sum(1, keepdims=True) + alpha * dim)
            kl = np.sum(desired * np.log2(desired / q), axis=1)
            score = (1.0 - lam) * rel - lam * kl
            score[~alive] = -np.inf
            j = int(np.argmax(score))
            out[u, t] = cands[j]
            alive[j] = False
            counts = counts + feats[j]
    return out


def user_fair_lists(base, diversified, advantaged):
    """Keep the base list for the lower group. MMR list for the group ahead."""
    out = base.copy()
    out[advantaged] = diversified[advantaged]
    return out


_DISC = 1.0 / np.log2(np.arange(1, K + 1) + 1.0)


def evaluate(index_lists, users, truth, profiles):
    """Accuracy, ILD, novelty, coverage, genre KL, popularity-band KL, gender NDCG."""
    n = len(users)
    ndcg = np.zeros(n)
    recall = np.zeros(n)
    ild = np.zeros(n)
    novelty = np.zeros(n)
    head = np.zeros(n)
    mid = np.zeros(n)
    tail = np.zeros(n)
    cal_kl = np.zeros(n)
    item_kl = np.zeros(n)
    shown = set()
    dim = len(GENRE_NAMES)
    for i, user in enumerate(users):
        cols = index_lists[i]
        rec = [ITEMS[j] for j in cols]
        shown.update(rec)
        relevant = truth[user]
        hits = np.fromiter((1.0 if it in relevant else 0.0 for it in rec), float, K)
        ideal = _DISC[:min(len(relevant), K)].sum()
        ndcg[i] = float((hits * _DISC).sum() / ideal) if ideal else 0.0
        recall[i] = float(hits.sum() / len(relevant))
        genres = [GENRES.get(it, frozenset()) for it in rec]
        dists = []
        for a in range(K):
            for b in range(a + 1, K):
                union = len(genres[a] | genres[b])
                dists.append(1.0 - len(genres[a] & genres[b]) / union if union else 0.0)
        ild[i] = float(np.mean(dists))
        surprisal = [-np.log2(POP_PROB[it]) for it in rec if POP_PROB.get(it, 0) > 0]
        novelty[i] = float(np.mean(surprisal)) if surprisal else 0.0
        bands = GROUP_INDEX[cols]
        head[i] = float(np.mean(bands == 0))
        mid[i] = float(np.mean(bands == 1))
        tail[i] = float(np.mean(bands == 2))
        q_genre = _smoothed(GENRE_DIST[cols].sum(0), ALPHA_GENRE)
        cal_kl[i] = float(np.sum(profiles[i] * np.log2(profiles[i] / q_genre)))
        q_item = _smoothed(np.bincount(bands, minlength=3).astype(float), ALPHA_ITEM)
        item_kl[i] = float(np.sum(TARGET_ITEM * np.log2(TARGET_ITEM / q_item)))
    gender = np.array([GENDER[u] for u in users])
    ndcg_f = float(ndcg[gender == "F"].mean())
    ndcg_m = float(ndcg[gender == "M"].mean())
    return {
        "ndcg": float(ndcg.mean()),
        "recall": float(recall.mean()),
        "ild": float(ild.mean()),
        "novelty": float(novelty.mean()),
        "coverage": len(shown & CATALOG) / len(CATALOG),
        "cal_kl": float(cal_kl.mean()),
        "item_kl": float(item_kl.mean()),
        "head": float(head.mean()),
        "mid": float(mid.mean()),
        "tail": float(tail.mean()),
        "ndcg_F": ndcg_f,
        "ndcg_M": ndcg_m,
        "gap": abs(ndcg_f - ndcg_m),
        "_ndcg": ndcg,
        "_gender": gender,
    }

In [ ]:
# =============================================================================
# 3.1.2  Select λ on validation, rerank the test lists, write tables
# =============================================================================
METHOD_SPEC = [
    # name, column optimised, True if higher is better
    ("mmr", "ild", True),
    ("calibration", "cal_kl", False),
    ("item_fair", "item_kl", False),
    ("user_fair", "gap", False),
]


def select_lambda(records, metric, higher, base_value):
    """Best grid point inside the NDCG budget. None keeps the base list."""
    feasible = [r for r in records if r["ndcg"] >= r["base_ndcg"] - BUDGET]
    pool = feasible if feasible else [min(records, key=lambda r: r["lam"])]
    if higher:
        best = max(pool, key=lambda r: (r[metric], -r["lam"]))
        improved = best[metric] > base_value + 1e-12
    else:
        best = min(pool, key=lambda r: (r[metric], r["lam"]))
        improved = best[metric] < base_value - 1e-12
    return best if improved else None


def dump_lists(path, run_id, label, method, lam, users, raw, index_lists):
    recommendations = {}
    for i, user in enumerate(users):
        cols = index_lists[i]
        recommendations[str(user)] = {
            "items": [str(ITEMS[j]) for j in cols],
            "scores": [float(raw[i, j]) for j in cols],
        }
    payload = {
        "run_id": f"{label}__{method}",
        "base_run": run_id,
        "method": method,
        "lam": lam,
        "k": K,
        "pool": POOL,
        "recommendations": recommendations,
    }
    path.write_text(json.dumps(payload))


curve_rows = []
test_rows = []
rerank_dir = RUNS / "rerank"
rerank_dir.mkdir(exist_ok=True)
(RUNS / "tables").mkdir(exist_ok=True)

published = pd.read_csv(RUNS / "tables" / "table_2_2_selected.csv").set_index("label")

for run_id, label in MODELS:
    pack = SCORES[run_id]
    rel_v, pool_v = candidate_pool(pack["valid"])
    rel_t, pool_t = candidate_pool(pack["test"])
    base_v_idx = pool_v[:, :K]
    base_t_idx = pool_t[:, :K]
    base_v = evaluate(base_v_idx, USERS_VALID, GT_VALID, PROFILE_VALID)
    base_t = evaluate(base_t_idx, USERS_TEST, GT_TEST, PROFILE_TEST)
    for metric in ("ndcg@10", "ild@10"):
        key = "ndcg" if metric.startswith("ndcg") else "ild"
        gap = abs(base_t[key] - float(published.loc[label, metric]))
        assert gap < 1e-6, (label, metric, base_t[key], gap)

    adv_name = "M" if base_v["ndcg_M"] >= base_v["ndcg_F"] else "F"
    adv_mask_v = base_v["_gender"] == adv_name
    print(
        f"{label}: validation NDCG F {base_v['ndcg_F']:.4f} / M {base_v['ndcg_M']:.4f} "
        f"-> group ahead is {adv_name}"
    )

    built_v = {"base": base_v_idx}
    for lam in GRID:
        built_v[("mmr", lam)] = mmr_lists(pool_v, rel_v, lam)
        built_v[("calibration", lam)] = kl_lists(
            pool_v, rel_v, GENRE_DIST, PROFILE_VALID, lam, ALPHA_GENRE)
        built_v[("item_fair", lam)] = kl_lists(
            pool_v, rel_v, GROUP_DIST, TARGET_ITEM, lam, ALPHA_ITEM)
        built_v[("user_fair", lam)] = user_fair_lists(
            base_v_idx, built_v[("mmr", lam)], adv_mask_v)

    chosen = {}
    for name, metric, higher in METHOD_SPEC:
        records = []
        for lam in GRID:
            metrics = evaluate(built_v[(name, lam)], USERS_VALID, GT_VALID, PROFILE_VALID)
            row = {"label": label, "method": name, "lam": lam, "base_ndcg": base_v["ndcg"]}
            row.update({k: v for k, v in metrics.items() if not k.startswith("_")})
            records.append(row)
            curve_rows.append(row)
        zero = {"label": label, "method": name, "lam": 0.0, "base_ndcg": base_v["ndcg"]}
        zero.update({k: v for k, v in base_v.items() if not k.startswith("_")})
        curve_rows.append(zero)
        best = select_lambda(records, metric, higher, base_v[metric])
        chosen[name] = 0.0 if best is None else best["lam"]
        moved = "kept" if best is None else f"λ={best['lam']:.2f}"
        print(f"  {name:<12} {moved}")

    gender_t = np.array([GENDER[u] for u in USERS_TEST])
    adv_mask_t = gender_t == adv_name
    test_lists = {
        "base": base_t_idx,
        "mmr": mmr_lists(pool_t, rel_t, chosen["mmr"]),
        "calibration": kl_lists(pool_t, rel_t, GENRE_DIST, PROFILE_TEST, chosen["calibration"], ALPHA_GENRE),
        "item_fair": kl_lists(pool_t, rel_t, GROUP_DIST, TARGET_ITEM, chosen["item_fair"], ALPHA_ITEM),
        "user_fair": user_fair_lists(
            base_t_idx, mmr_lists(pool_t, rel_t, chosen["user_fair"]), adv_mask_t),
    }
    for name, _metric, _higher in [("base", None, None), *METHOD_SPEC]:
        lam = 0.0 if name == "base" else chosen[name]
        metrics = base_t if name == "base" else evaluate(
            test_lists[name], USERS_TEST, GT_TEST, PROFILE_TEST)
        n_changed = int(np.any(test_lists[name] != base_t_idx, axis=1).sum())
        row = {
            "label": label, "method": name, "lam": lam, "adv_group": adv_name,
            "lists_changed": n_changed,
        }
        row.update({k: v for k, v in metrics.items() if not k.startswith("_")})
        test_rows.append(row)
        if name != "base":
            dump_lists(
                rerank_dir / f"{label}__{name}_top10.json",
                run_id, label, name, lam, USERS_TEST, pack["test"], test_lists[name],
            )

CURVES = pd.DataFrame(curve_rows)
TEST = pd.DataFrame(test_rows)
CURVES.to_csv(RUNS / "tables" / "table_3_1_validation.csv", index=False)
TEST.to_csv(RUNS / "tables" / "table_3_1_test.csv", index=False)

_order = ["base", "mmr", "calibration", "item_fair", "user_fair"]
TEST["method"] = pd.Categorical(TEST["method"], _order, ordered=True)
TEST = TEST.sort_values(["label", "method"]).reset_index(drop=True)

_acc_cols = ["label", "method", "lam", "ndcg", "recall", "ild", "novelty", "coverage", "lists_changed"]
_fair_cols = ["label", "method", "lam", "cal_kl", "item_kl", "head", "mid", "tail", "ndcg_F", "ndcg_M", "gap"]
with pd.option_context("display.float_format", "{:.4f}".format):
    print("\ntest accuracy and diversity")
    print(TEST[_acc_cols].to_string(index=False))
    print("\ntest calibration and fairness")
    print(TEST[_fair_cols].to_string(index=False))

# Validation trade-off for EASE, the reference accurate model. The vertical line is the λ that was kept.
fig, axes = plt.subplots(2, 2, figsize=(11, 7.2))
_ylabels = {
    "mmr": "ILD@10",
    "calibration": "genre KL (bits)",
    "item_fair": "exposure KL (bits)",
    "user_fair": "|NDCG gap|",
}
_titles = {
    "mmr": "Diversification",
    "calibration": "Calibration",
    "item_fair": "Item-side fairness",
    "user_fair": "User-side fairness",
}
ease_lam = {r["method"]: r["lam"] for _, r in TEST[TEST["label"] == "EASE"].iterrows()}
for ax, (name, _metric, _higher) in zip(axes.ravel(), METHOD_SPEC):
    sub = CURVES[(CURVES["label"] == "EASE") & (CURVES["method"] == name)].sort_values("lam")
    line = ax.plot(sub["lam"], sub[_metric], color="#e6550d", marker="o", label=_ylabels[name])
    ax.set_xlabel("λ")
    ax.set_ylabel(_ylabels[name], color="#e6550d")
    twin = ax.twinx()
    twin.plot(sub["lam"], sub["ndcg"], color="#08519c", marker="s", label="NDCG@10")
    twin.set_ylabel("NDCG@10", color="#08519c")
    ax.axvline(ease_lam[name], color="#333333", ls="--", lw=1)
    ax.set_title(_titles[name])
    ax.grid(alpha=0.3)
fig.suptitle("Task 3.1 — validation trade-off on EASE (dashed line is the λ used at test)")
fig.tight_layout()
fig.savefig(RUNS / "tables" / "fig_task31.png", dpi=130, bbox_inches="tight")
plt.show()
print("saved runs/tables/table_3_1_validation.csv, table_3_1_test.csv, fig_task31.png")
print(f"saved {len(list(rerank_dir.glob('*_top10.json')))} reranked lists in runs/rerank/")

### Reading Task 3.1

λ is chosen on validation, inside a drop of 0.02 NDCG. EASE and SLIMElastic take 0.3 for MMR, 0.3 for calibration, 0.5 for item-side fairness and 0.4 for user-side fairness. NGCF takes 0.2. NeuMF and BPR take smaller values, because their validation NDCG is already lower and the same budget is spent sooner. On the test split the drop can run past 0.02: EASE item-side fairness goes from 0.330 to 0.303.

MMR rewrites all 943 lists and is what moves ILD off the flat band from Task 2.1. EASE goes from 0.810 to 0.883, and the other four land between 0.875 and 0.885. The cost on EASE is NDCG from 0.330 to 0.313. Calibration cuts the genre KL by about half (EASE from 0.808 to 0.465) and barely moves ILD. On EASE the recall stays at 0.278 against 0.281, so the NDCG drop is mostly hits sitting lower in the list.

Item-side fairness is the reranker that moves the popularity bands. On EASE the head share falls from 0.876 to 0.636 and the tail share from 0.005 to 0.107, with coverage from 0.237 to 0.365. Men are ahead on validation for every model, so user-side fairness rewrites their 670 lists and leaves the women on the base list. The EASE gender gap falls from 0.044 to 0.015, and the women's NDCG stays at 0.299. The mean NDCG is lower after this rerank. The method spends accuracy the higher group already had.
